# Automated Financial Customer Complaint Classification
**Shorya Jaiswal — NLP project**

This notebook accompanies the complete source in `src/project.py`. Read `REPORT.md` for methodology. Included outputs come from real public CFPB narratives; the style audit uses proxies pending human annotation.

## 1. Setup
Extract the complete ZIP. Open this notebook from its folder. Install `requirements.txt` first. For Colab, upload and extract the complete ZIP, then change the working directory to `financial_complaints`.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT = Path.cwd()
assert (ROOT / "src/project.py").exists(), "Open from the extracted financial_complaints folder"
sys.path.insert(0, str(ROOT / "src"))
from project import clean, style, load_data
print("Project:", ROOT.name)

Project: financial_complaints


## 2. Source and sample
The API result window is not a random sample of all complaints. Observe product imbalance before interpreting accuracy.

In [2]:
print(Path("data/provenance.json").read_text())
raw = pd.read_csv("data/complaints.csv")
print(raw.shape)
print(raw["Product"].value_counts().to_string())

{
  "url": "https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/?size=10000&no_aggs=true&has_narrative=true",
  "retrieved_utc": "2026-09-13T19:21:19.605100+00:00",
  "rows": 10000,
  "selection": "First API result window under server default ordering; not a random database sample.",
  "date_range": [
    "2024-08-26T08:44:50.000Z",
    "2024-09-04T07:16:19.000Z"
  ]
}
(10000, 4)
Product
Credit reporting or other personal consumer reports        8299
Debt collection                                             692
Credit card                                                 309
Checking or savings account                                 300
Mortgage                                                     97
Money transfer, virtual currency, or money service           91
Vehicle loan or lease                                        67
Student loan                                                 65
Payday loan, title loan, personal loan, or advance loan      43
Prepa

## 3. Cleaning and label preparation
No target or structured complaint fields enter the feature matrix. Deduplication happens before splitting.

In [3]:
data, manifest = load_data("data/complaints.csv", 30000)
print(json.dumps(manifest, indent=2))
print(data.groupby("label").size().to_string())
print("Cleaned example:", clean("My mortgage payment was NOT credited! XXXX 123"))

{
  "source_rows": 10000,
  "eligible_rows": 10000,
  "sample_limit": 30000,
  "retained": 5042,
  "conflicting_rows_removed": 190,
  "rare_classes_excluded": {
    "Prepaid cards": 22,
    "Debt/credit management": 15
  },
  "unmapped_products": []
}
label
Bank accounts                   300
Credit cards                    294
Credit reporting               3600
Debt collection                 488
Money services                   90
Mortgages                        97
Payday/title/personal loans      42
Student loans                    64
Vehicle loans                    67
Cleaned example: my mortgage payment was not credited


## 4. Train and select models
The code below can rerun the entire experiment. Leave RUN_TRAINING=False to inspect the included measured results. Training fits TF-IDF on the training split; validation macro-F1 selects settings; the frozen winner is evaluated on the final test split.

In [4]:
RUN_TRAINING = False
if RUN_TRAINING:
    from argparse import Namespace
    from project import train
    train(Namespace(csv="data/complaints.csv", output="outputs", max_rows=30000, max_features=30000))
print(pd.read_csv("outputs/validation_comparison.csv").to_string(index=False))

                  model   n  accuracy  macro_f1  weighted_f1
         Linear SVM C=1 756  0.882275  0.673661     0.876926
       Linear SVM C=0.5 756  0.882275  0.672406     0.878317
Logistic Regression C=1 756  0.847884  0.646250     0.852304
Logistic Regression C=4 756  0.863757  0.635433     0.863749
          Random Forest 756  0.850529  0.529397     0.837677
      Majority baseline 756  0.714286  0.092593     0.595238


## 5. Final test performance
Accuracy alone is insufficient for this imbalanced sample. Review macro-F1 and each product’s recall.

In [5]:
print(Path("outputs/test_metrics.json").read_text())
print(pd.read_csv("outputs/classification_report.csv").to_string(index=False))

{
  "n": 757,
  "accuracy": 0.8890356671070013,
  "macro_f1": 0.68335623257988,
  "weighted_f1": 0.8842265981366503,
  "selected_model": "Linear SVM C=1",
  "accuracy_ci": [
    0.8665785997357992,
    0.9101717305151915
  ]
}
                 Unnamed: 0  precision   recall  f1-score    support
              Bank accounts   0.720000 0.800000  0.757895  45.000000
               Credit cards   0.714286 0.795455  0.752688  44.000000
           Credit reporting   0.934901 0.957407  0.946020 540.000000
            Debt collection   0.779412 0.716216  0.746479  74.000000
             Money services   0.818182 0.642857  0.720000  14.000000
                  Mortgages   1.000000 0.800000  0.888889  15.000000
Payday/title/personal loans   0.000000 0.000000  0.000000   6.000000
              Student loans   0.857143 0.666667  0.750000   9.000000
              Vehicle loans   0.714286 0.500000  0.588235  10.000000
                   accuracy   0.889036 0.889036  0.889036   0.889036
              

## 6. Audit: length and style proxies
Short ≤50 words; long ≥200 words. Formal/informal proxies additionally require predefined textual markers. Proxy groups are not validated formality labels. Product-matched results reduce composition differences but are not causal evidence.

In [6]:
print(pd.read_csv("outputs/audit.csv").to_string(index=False))
print(Path("outputs/controlled_audit.json").read_text())

  n  accuracy  macro_f1  weighted_f1    dimension                 group  accuracy_ci_low  accuracy_ci_high  small_group
263  0.889734  0.684361     0.887411 length_group                  long         0.851711          0.923954        False
376  0.904255  0.670201     0.900623 length_group                medium         0.872340          0.930851        False
118  0.838983  0.598652     0.819575 length_group                 short         0.771186          0.898305        False
113  0.858407  0.699510     0.856948  style_proxy formal_detailed_proxy         0.778761          0.911504        False
  5  0.800000  0.555556     0.733333  style_proxy  informal_short_proxy         0.400000          1.000000         True
639  0.895149  0.667820     0.889656  style_proxy            unassigned         0.870070          0.917097        False
[
  {
    "dimension": "length_group",
    "group_1": "short",
    "group_2": "long",
    "eligible_classes": 4,
    "class_results": [
      [
        "Bank ac

## 7. Charts
![Validation comparison](outputs/model_comparison.png)

![Confusion matrix](outputs/confusion_matrix.png)

![Length audit](outputs/length_audit.png)

## 8. Explainability and live prediction
Positive TF-IDF × coefficient contributions are contributions to a linear score, not probabilities or SHAP values.

In [7]:
print(pd.read_csv("outputs/top_features.csv").groupby("class").head(5).to_string(index=False))
from project import predict
from argparse import Namespace
predict(Namespace(model="outputs/model.joblib", text="My mortgage servicer charged a late fee after I made my monthly payment."))

                      class             term  coefficient explanation_model
              Bank accounts         checking     1.882136    Linear SVM C=1
              Bank accounts             bank     1.866128    Linear SVM C=1
              Bank accounts            chime     1.723523    Linear SVM C=1
              Bank accounts       my account     1.722434    Linear SVM C=1
              Bank accounts checking account     1.397745    Linear SVM C=1
               Credit cards             card     3.043131    Linear SVM C=1
               Credit cards      credit card     2.271892    Linear SVM C=1
               Credit cards         interest     1.527442    Linear SVM C=1
               Credit cards         the card     1.170261    Linear SVM C=1
               Credit cards        synchrony     1.107581    Linear SVM C=1
           Credit reporting            these     1.585866    Linear SVM C=1
           Credit reporting          equifax     1.492507    Linear SVM C=1
           C

## 9. Error analysis
Read misclassified cases and record your own explanation. Do not assume every error is caused by writing style.

In [8]:
errors = pd.read_csv("outputs/error_examples.csv")
print(errors[["label", "prediction", "word_count"]].head(10).to_string(index=False))

           label       prediction  word_count
 Debt collection Credit reporting          15
 Debt collection Credit reporting          46
    Credit cards    Bank accounts          93
 Debt collection Credit reporting          66
 Debt collection Credit reporting          81
 Debt collection Credit reporting          76
Credit reporting  Debt collection          26
Credit reporting     Credit cards         259
Credit reporting  Debt collection         153
 Debt collection Credit reporting          36


## 10. Human validation
Complete `outputs/human_audit_template.csv` according to the README. Two independent readers label formality, then adjudicate. Use `python src/project.py human-audit` afterwards. Human annotations have deliberately not been fabricated.

## 11. Conclusion
Consult `outputs/RESULTS.md` for measured findings and `REPORT.md` for the full discussion. This is a sample-specific routing prototype; test accuracy does not establish performance on the full CFPB database or in production.